# 03 · Piloto: dataset 128×128 y prueba corta de las 3 técnicas

**Fase CRISP-DM:** 4. Modelado (preparación del modelado).
**Objetivo:** antes de gastar unidades en entrenamientos largos, medir en la GPU real:
- velocidad (imágenes/s, s/kimg), memoria de GPU y si todo funciona,
- costo estimado (horas y unidades) de cada entrenamiento completo.

**Técnicas (todas desde cero, condicionales por clase, 128×128):**
1. **VAE condicional** (implementación propia en PyTorch).
2. **Difusión condicional** (`UNet2DModel` de `diffusers` + scheduler DDPM; muestreo DDIM).
3. **GAN condicional:** StyleGAN2-ADA, repositorio oficial de NVIDIA (`--cond=1`).

**Reglas que respeta este notebook**
- Solo se usan imágenes de **train**. Val se prepara a 128 para la selección de checkpoints en notebooks posteriores. **Test no se toca.**
- **Sin volteo horizontal** (`--mirror=0` y sin flips en VAE/difusión): los productos tienen texto y logos; un flip produce empaques que no existen y dañaría la fidelidad semántica.
- Las métricas internas de StyleGAN2-ADA se desactivan (`--metrics=none`): calculan FID con Inception contra el set de entrenamiento; la selección se hará con DINOv2 sobre val.

**Entorno:** GPU **L4**, versión de entorno fija (**2026.07**). Duración total ≈ 45–60 min.

## Paso 1 · Entorno y GPU

In [2]:
import sys, os, subprocess, platform
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
import torch
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA (torch):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ SIN GPU")
if torch.cuda.is_available():
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
          "| bf16:", torch.cuda.is_bf16_supported())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("⚠️ Este piloto necesita GPU: Entorno de ejecución → Cambiar tipo → GPU L4")

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | CUDA (torch): 12.8
GPU: NVIDIA L4
VRAM total (GB): 22.0 | bf16: True


## Paso 2 · Parámetros del piloto
Puedes acortar los tiempos si quieres gastar menos. `UNITS_PER_HOUR`: cópialo del panel de recursos de Colab (lo que consume la L4 por hora).

In [3]:
SEED = 42
RES = 128
UNITS_PER_HOUR = 1.71            # ← reemplázalo por la tasa que muestra tu panel de recursos

PILOT_MIN_VAE = float(os.environ.get("PILOT_MIN_VAE", 5))      # minutos de entrenamiento medido
PILOT_MIN_DIFF = float(os.environ.get("PILOT_MIN_DIFF", 10))
PILOT_MIN_GAN = float(os.environ.get("PILOT_MIN_GAN", 15))
BATCH_VAE = int(os.environ.get("BATCH_VAE", 64))
BATCH_DIFF = int(os.environ.get("BATCH_DIFF", 32))
DDIM_STEPS = int(os.environ.get("DDIM_STEPS", 50))
N_SAMPLE_TIMING = int(os.environ.get("N_SAMPLE_TIMING", 34))   # 2 imágenes por clase para medir el muestreo

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## Paso 3 · Drive, rutas y verificación de la partición congelada

In [4]:
import json, shutil, hashlib, zipfile, time, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()

DIRS = {k: PROJECT_DIR / k for k in ["manifest", "splits", "logs", "datasets", "eda", "eda/piloto", "checkpoints"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG02 = json.loads((DIRS["logs"] / "02_preparacion.json").read_text())
SPLIT_ZIP_DRIVE = DIRS["splits"] / "grocpos_split_v1.zip"
SPLIT_ZIP_LOCAL = LOCAL_DIR / "grocpos_split_v1.zip"
if not SPLIT_ZIP_LOCAL.exists():
    shutil.copy2(SPLIT_ZIP_DRIVE, SPLIT_ZIP_LOCAL)
assert sha256_file(SPLIT_ZIP_LOCAL) == LOG02["zip_split_sha256"], "⚠️ El zip de partición no coincide con el del notebook 02"
SPLIT_DIR = LOCAL_DIR / "grocpos_split_v1"
if not SPLIT_DIR.exists():
    with zipfile.ZipFile(SPLIT_ZIP_LOCAL) as zf:
        zf.extractall(SPLIT_DIR)

split = pd.read_csv(DIRS["splits"] / "split_v1.csv")
CLASS_MAP = json.loads((DIRS["manifest"] / "class_map.json").read_text())
N_CLASSES = len(CLASS_MAP)
split["label"] = split["clase"].map(CLASS_MAP)
assert split["label"].notna().all()
print("Partición verificada ✔ |", split["split"].value_counts().to_dict(), "| clases:", N_CLASSES)

Mounted at /content/drive
Partición verificada ✔ | {'train': 3060, 'test': 1530, 'val': 510} | clases: 17


## Paso 4 · Dataset 128×128 (train y val)

- Redimensionado 640→128 con filtro **Lanczos** (antialiasing). Las imágenes ya venían estiradas a 640×640 por Roboflow, así que no se recortan.
- Se guardan como PNG (sin pérdida) en el disco local y como zip en Drive (`datasets/`), con su SHA-256.
- Para StyleGAN2-ADA se crea además un zip **sin compresión** con `dataset.json` (formato `{"labels": [[archivo, clase], ...]}`, verificado en `training/dataset.py` del repo oficial).

In [5]:
from PIL import Image

def preparar(particion):
    out = LOCAL_DIR / f"res{RES}" / particion
    sub = split[split["split"] == particion].sort_values("image_id").reset_index(drop=True)
    if out.exists():
        shutil.rmtree(out)
    rutas = []
    for _, r in sub.iterrows():
        src = next((SPLIT_DIR / particion / r["clase"]).glob(f"{r['image_id']}.*"))
        dst = out / f"{int(r['label']):02d}" / f"{r['image_id']}.png"
        dst.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(src) as im:
            im.convert("RGB").resize((RES, RES), Image.LANCZOS).save(dst)
        rutas.append(dst.relative_to(out).as_posix())
    sub["png"] = rutas
    return out, sub

t0 = time.time()
TRAIN_DIR, train_df = preparar("train")
VAL_DIR, val_df = preparar("val")
print(f"train: {len(train_df)} | val: {len(val_df)} | {time.time()-t0:.0f}s")

def zip_carpeta(carpeta, df, destino, con_labels):
    with zipfile.ZipFile(destino, "w", compression=zipfile.ZIP_STORED) as zf:
        for p in df["png"]:
            zf.write(carpeta / p, arcname=p)
        if con_labels:
            labels = [[p, int(l)] for p, l in zip(df["png"], df["label"])]
            zf.writestr("dataset.json", json.dumps({"labels": labels}))
    return sha256_file(destino)

SG_ZIP = LOCAL_DIR / f"train{RES}_stylegan.zip"
SHA_TRAIN = zip_carpeta(TRAIN_DIR, train_df, SG_ZIP, con_labels=True)
VAL_ZIP = LOCAL_DIR / f"val{RES}.zip"
SHA_VAL = zip_carpeta(VAL_DIR, val_df, VAL_ZIP, con_labels=True)
shutil.copy2(SG_ZIP, DIRS["datasets"] / SG_ZIP.name)
shutil.copy2(VAL_ZIP, DIRS["datasets"] / VAL_ZIP.name)
train_df.to_csv(DIRS["datasets"] / f"train{RES}_index.csv", index=False)
val_df.to_csv(DIRS["datasets"] / f"val{RES}_index.csv", index=False)
print("SHA-256 train:", SHA_TRAIN, "\nSHA-256 val:  ", SHA_VAL)

train: 3060 | val: 510 | 47s
SHA-256 train: efa4c501ef649a86585da74db94474477edf0fbd230a9aa4aa3dd10a0188e15a 
SHA-256 val:   d81280a5054de351dfed01a11297dca4c6d07194554392a9bb72a8b2f70c9dc1


## Paso 5 · Cargar train en memoria de GPU
3060 imágenes 128×128 en uint8 ocupan ~150 MB: caben en la GPU y evitan cuellos de botella de lectura.

In [6]:
import torch.nn as nn
import torch.nn.functional as F

X_train = torch.from_numpy(np.stack([np.asarray(Image.open(TRAIN_DIR / p)) for p in train_df["png"]]))
X_train = X_train.permute(0, 3, 1, 2).contiguous().to(DEVICE)                 # uint8 [N,3,128,128]
Y_train = torch.tensor(train_df["label"].to_numpy(), dtype=torch.long, device=DEVICE)
print(X_train.shape, X_train.dtype, "| etiquetas:", Y_train.unique().numel())

GEN = torch.Generator(device=DEVICE).manual_seed(SEED)
def lote(bs):
    idx = torch.randint(0, X_train.shape[0], (bs,), device=DEVICE, generator=GEN)
    x = X_train[idx].float() / 127.5 - 1.0                                   # [-1, 1]
    return x, Y_train[idx]

AMP = dict(device_type=DEVICE.type, dtype=torch.bfloat16,
           enabled=(DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) or DEVICE.type == "cpu")

def medir(paso, minutos, bs, warmup=20):
    # Ejecuta 'paso' durante 'minutos' (tras un calentamiento) y devuelve métricas de velocidad.
    for _ in range(warmup):
        paso(bs)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0, n, perdidas = time.time(), 0, []
    while time.time() - t0 < minutos * 60:
        perdidas.append(paso(bs)); n += 1
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    dt = time.time() - t0
    return {"pasos": n, "segundos": round(dt, 1), "pasos_por_s": round(n / dt, 3),
            "img_por_s": round(n * bs / dt, 1), "batch": bs,
            "vram_pico_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if DEVICE.type == "cuda" else None,
            "perdida_inicio": round(float(np.mean(perdidas[:10])), 4),
            "perdida_final": round(float(np.mean(perdidas[-10:])), 4)}

def guardar_malla(x, nombre, nrow=17):
    import torchvision.utils as vutils
    x = (x.float().clamp(-1, 1) + 1) / 2
    ruta = DIRS["eda/piloto"] / nombre
    vutils.save_image(x, ruta, nrow=nrow)
    return ruta

RESULTADOS = {}

torch.Size([3060, 3, 128, 128]) torch.uint8 | etiquetas: 17


## Paso 6 · Piloto VAE condicional

Encoder convolucional 128→4 y decoder simétrico; la clase entra como *embedding* en el encoder y en el decoder.
Pérdida: reconstrucción (MSE) + β·KL, con β = 1. Es una arquitectura de partida; se puede ajustar en el notebook de entrenamiento.

In [7]:
LATENTE = 256

def bloque_down(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 4, 2, 1), nn.GroupNorm(8, cout), nn.SiLU())
def bloque_up(cin, cout):
    return nn.Sequential(nn.ConvTranspose2d(cin, cout, 4, 2, 1), nn.GroupNorm(8, cout), nn.SiLU())

class CVAE(nn.Module):
    def __init__(self, n_clases, z=LATENTE, emb=64):
        super().__init__()
        self.emb = nn.Embedding(n_clases, emb)
        self.enc = nn.Sequential(bloque_down(3, 64), bloque_down(64, 128), bloque_down(128, 256),
                                 bloque_down(256, 256), bloque_down(256, 512))      # 128 → 4
        self.mu = nn.Linear(512 * 16 + emb, z)
        self.logvar = nn.Linear(512 * 16 + emb, z)
        self.fc = nn.Linear(z + emb, 512 * 16)
        self.dec = nn.Sequential(bloque_up(512, 256), bloque_up(256, 256), bloque_up(256, 128),
                                 bloque_up(128, 64), nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
    def encode(self, x, y):
        h = torch.cat([self.enc(x).flatten(1), self.emb(y)], 1)
        return self.mu(h), self.logvar(h)
    def decode(self, z, y):
        h = self.fc(torch.cat([z, self.emb(y)], 1)).view(-1, 512, 4, 4)
        return self.dec(h)
    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decode(z, y), mu, logvar

torch.manual_seed(SEED)
vae = CVAE(N_CLASSES).to(DEVICE)
opt_vae = torch.optim.AdamW(vae.parameters(), lr=2e-4)
print("Parámetros VAE (M):", round(sum(p.numel() for p in vae.parameters()) / 1e6, 1))

def paso_vae(bs, beta=1.0):
    x, y = lote(bs)
    with torch.autocast(**AMP):
        xr, mu, logvar = vae(x, y)
    rec = F.mse_loss(xr.float(), x, reduction="sum") / bs
    kl = (-0.5 * (1 + logvar.float() - mu.float() ** 2 - logvar.float().exp()).sum()) / bs
    loss = rec + beta * kl
    opt_vae.zero_grad(set_to_none=True); loss.backward(); opt_vae.step()
    return loss.item()

RESULTADOS["vae"] = medir(paso_vae, PILOT_MIN_VAE, BATCH_VAE)
RESULTADOS["vae"]["parametros_M"] = round(sum(p.numel() for p in vae.parameters()) / 1e6, 1)
vae.eval()
with torch.no_grad(), torch.autocast(**AMP):
    y = torch.arange(N_CLASSES, device=DEVICE).repeat(2)
    muestras = vae.decode(torch.randn(len(y), LATENTE, device=DEVICE), y)
guardar_malla(muestras, "vae_piloto.png", nrow=N_CLASSES)
RESULTADOS["vae"]

Parámetros VAE (M): 14.5


{'pasos': 8383,
 'segundos': 300.0,
 'pasos_por_s': 27.943,
 'img_por_s': 1788.3,
 'batch': 64,
 'vram_pico_gb': 1.1,
 'perdida_inicio': 13394.6385,
 'perdida_final': 1357.4357,
 'parametros_M': 14.5}

## Paso 7 · Piloto difusión condicional (desde cero)

- `UNet2DModel` de `diffusers` con `num_class_embeds` (verificado en la documentación: el *embedding* de clase se suma al del paso temporal).
- U-Net de tamaño moderado: canales (64, 128, 256, 256, 512), atención en 16×16 y 8×8.
- Se reserva un índice extra como **clase nula** (10 % de los pasos) para permitir *classifier-free guidance* al generar.
- Entrenamiento con scheduler DDPM (1000 pasos, coseno); el muestreo se mide con **DDIM** a `DDIM_STEPS` pasos.

In [8]:
%pip install -q diffusers

In [9]:
import diffusers
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler

torch.manual_seed(SEED)
unet = UNet2DModel(
    sample_size=RES, in_channels=3, out_channels=3, layers_per_block=2,
    block_out_channels=(64, 128, 256, 256, 512),
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"),
    num_class_embeds=N_CLASSES + 1,                    # +1 = clase nula para classifier-free guidance
).to(DEVICE)
NULL = N_CLASSES
P_UNCOND = 0.1
sched = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")
opt_unet = torch.optim.AdamW(unet.parameters(), lr=1e-4)
print("diffusers:", diffusers.__version__, "| Parámetros U-Net (M):",
      round(sum(p.numel() for p in unet.parameters()) / 1e6, 1))

def paso_diff(bs):
    x, y = lote(bs)
    y = torch.where(torch.rand(bs, device=DEVICE) < P_UNCOND, torch.full_like(y, NULL), y)
    ruido = torch.randn_like(x)
    t = torch.randint(0, sched.config.num_train_timesteps, (bs,), device=DEVICE)
    xt = sched.add_noise(x, ruido, t)
    with torch.autocast(**AMP):
        pred = unet(xt, t, class_labels=y).sample
    loss = F.mse_loss(pred.float(), ruido)
    opt_unet.zero_grad(set_to_none=True); loss.backward(); opt_unet.step()
    return loss.item()

RESULTADOS["difusion"] = medir(paso_diff, PILOT_MIN_DIFF, BATCH_DIFF)
RESULTADOS["difusion"]["parametros_M"] = round(sum(p.numel() for p in unet.parameters()) / 1e6, 1)
RESULTADOS["difusion"]

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


diffusers: 0.39.0 | Parámetros U-Net (M): 72.7


{'pasos': 947,
 'segundos': 600.0,
 'pasos_por_s': 1.578,
 'img_por_s': 50.5,
 'batch': 32,
 'vram_pico_gb': 11.48,
 'perdida_inicio': 0.2584,
 'perdida_final': 0.0559,
 'parametros_M': 72.7}

In [10]:
# Velocidad de muestreo (DDIM + classifier-free guidance). Con pocos minutos de entrenamiento las imágenes serán ruido: solo medimos tiempo.
@torch.no_grad()
def muestrear(y, pasos=DDIM_STEPS, guia=2.0):
    ddim = DDIMScheduler.from_config(sched.config)
    ddim.set_timesteps(pasos)
    x = torch.randn(len(y), 3, RES, RES, device=DEVICE)
    y_null = torch.full_like(y, NULL)
    for t in ddim.timesteps:
        with torch.autocast(**AMP):
            e = unet(torch.cat([x, x]), t, class_labels=torch.cat([y, y_null])).sample.float()
        e_c, e_u = e.chunk(2)
        x = ddim.step(e_u + guia * (e_c - e_u), t, x).prev_sample
    return x

unet.eval()
y = torch.arange(N_CLASSES, device=DEVICE).repeat_interleave(max(1, N_SAMPLE_TIMING // N_CLASSES))
if DEVICE.type == "cuda": torch.cuda.synchronize()
t0 = time.time(); muestras = muestrear(y)
if DEVICE.type == "cuda": torch.cuda.synchronize()
seg = time.time() - t0
RESULTADOS["difusion"].update({"muestreo_ddim_pasos": DDIM_STEPS, "muestreo_seg_por_img": round(seg / len(y), 3)})
guardar_malla(muestras, "difusion_piloto.png", nrow=N_CLASSES)
unet.train()
print(f"Muestreo: {seg/len(y):.3f} s/imagen con {DDIM_STEPS} pasos DDIM")

Muestreo: 0.754 s/imagen con 50 pasos DDIM


In [11]:
# Liberar memoria de GPU antes de StyleGAN2-ADA (corre en otro proceso)
del vae, opt_vae, unet, opt_unet, X_train
import gc; gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## Paso 8 · Piloto StyleGAN2-ADA (repositorio oficial, condicional)

- Se clona el repositorio oficial en un commit fijo (reproducibilidad).
- **Parches de compatibilidad** (detectados al probar el repo con Python 3.13 y PyTorch 2.x):
  1. `dnnlib/util.py` importa `strtobool` desde `distutils`, módulo que ya no existe en Python ≥ 3.12 → se reemplaza por una función equivalente.
  2. `torch_utils/misc.py` llama `Sampler.__init__(dataset)`, firma eliminada en PyTorch moderno → se llama sin argumentos (no cambia el comportamiento).
- Si las extensiones CUDA propias no compilan con el PyTorch actual, el repo **cae a una implementación de referencia** (más lenta) y lo avisa; esta celda detecta ese aviso.
- Configuración: `--cond=1 --mirror=0 --metrics=none --cfg=auto` (a 128 px y 1 GPU, `auto` usa batch 32).
- Se ejecuta durante `PILOT_MIN_GAN` minutos y se detiene; el log queda en Drive.

In [12]:
%pip install -q click requests tqdm ninja psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 23.0 MB/s eta 0:00:00


In [13]:
SG_COMMIT = "d72cc7d041b42ec8e806021a205ed9349f87c6a4"
SG_DIR = LOCAL_DIR / "stylegan2-ada-pytorch"
if not SG_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/NVlabs/stylegan2-ada-pytorch.git", str(SG_DIR)], check=True)
    subprocess.run(["git", "-C", str(SG_DIR), "checkout", "-q", SG_COMMIT], check=True)
commit = subprocess.run(["git", "-C", str(SG_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert commit == SG_COMMIT, commit

util = SG_DIR / "dnnlib" / "util.py"
src = util.read_text()
if "from distutils.util import strtobool" in src:
    src = src.replace(
        "from distutils.util import strtobool",
        "try:\n    from distutils.util import strtobool\nexcept ImportError:  # Python >= 3.12\n"
        "    def strtobool(v):\n        return v.lower() in ('y', 'yes', 't', 'true', 'on', '1')")
    util.write_text(src)

# Parche 2: en PyTorch moderno, torch.utils.data.Sampler.__init__ ya no acepta argumentos
misc = SG_DIR / "torch_utils" / "misc.py"
src = misc.read_text()
src = src.replace("        super().__init__(dataset)\n", "        super().__init__()\n")
misc.write_text(src)
assert "super().__init__(dataset)" not in misc.read_text()
print("Repo en commit", commit[:10], "| parches distutils y Sampler aplicados ✔")

Repo en commit d72cc7d041 | parches distutils y Sampler aplicados ✔


In [14]:
GAN_RUNS = LOCAL_DIR / "sg_piloto_runs"
if GAN_RUNS.exists(): shutil.rmtree(GAN_RUNS)
GAN_LOG = DIRS["logs"] / "03_piloto_stylegan.log"
cmd = [sys.executable, "train.py", f"--outdir={GAN_RUNS}", f"--data={SG_ZIP}", "--gpus=1",
       "--cond=1", "--mirror=0", "--metrics=none", "--cfg=auto", "--kimg=1000", "--snap=1000", f"--seed={SEED}"]
print(" ".join(cmd))

t0 = time.time()
with open(GAN_LOG, "w") as flog:
    proc = subprocess.Popen(cmd, cwd=SG_DIR, stdout=flog, stderr=subprocess.STDOUT, text=True)
    try:
        proc.wait(timeout=PILOT_MIN_GAN * 60)
    except subprocess.TimeoutExpired:
        proc.terminate()
        try: proc.wait(timeout=30)
        except subprocess.TimeoutExpired: proc.kill()
dur = time.time() - t0
log_txt = GAN_LOG.read_text(errors="ignore")
print(f"Ejecutó {dur/60:.1f} min | código de salida: {proc.returncode}")
print("\n".join(log_txt.splitlines()[-25:]))

/usr/bin/python3 train.py --outdir=/content/data/sg_piloto_runs --data=/content/data/train128_stylegan.zip --gpus=1 --cond=1 --mirror=0 --metrics=none --cfg=auto --kimg=1000 --snap=1000 --seed=42
Ejecutó 2.9 min | código de salida: 1
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1353, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 907, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/decorators.py", line 34, in new_func
    return f(get_current_context(), *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/data/stylegan2-ada-pytorch/train.py", line 531, in main
    subprocess_fn(rank=0, args=args, temp_dir=temp_dir)
  File "/content/data/stylegan2-ada-pytorch/train.py", line 383, in subprocess_fn
    training_

In [15]:
ticks = [(int(a), float(b), float(c)) for a, b, c in
         re.findall(r"tick\s+(\d+)\s+kimg\s+([\d.]+).*?sec/kimg\s+([\d.]+)", log_txt)]
gpumem = [float(x) for x in re.findall(r"gpumem\s+([\d.]+)", log_txt)]
fallback = sorted(set(re.findall(r"Failed to build CUDA kernels for (\w+)", log_txt)))
gradfix = sorted(set(re.findall(r"(\w+_gradfix) not supported", log_txt)))
errores = re.findall(r"(?m)^\w*Error.*$", log_txt)

utiles = [s for t, k, s in ticks if t >= 1]              # el tick 0 incluye compilación e inicialización
RESULTADOS["gan"] = {
    "ticks_registrados": len(ticks),
    "kimg_alcanzado": ticks[-1][1] if ticks else 0.0,
    "sec_por_kimg": round(float(np.median(utiles)), 2) if utiles else None,
    "img_por_s": round(1000 / float(np.median(utiles)), 1) if utiles else None,
    "vram_pico_gb": max(gpumem) if gpumem else None,
    "kernels_cuda_fallback": fallback,
    "gradfix_desactivado": gradfix,
    "errores": errores[:5],
}
RESULTADOS["gan"]

{'ticks_registrados': 0,
 'kimg_alcanzado': 0.0,
 'sec_por_kimg': None,
 'img_por_s': None,
 'vram_pico_gb': None,
 'kernels_cuda_fallback': ['bias_act', 'upfirdn2d'],
 'gradfix_desactivado': ['conv2d_gradfix', 'grid_sample_gradfix'],
 'errores': ["ModuleNotFoundError: No module named 'bias_act_plugin'",
  "ModuleNotFoundError: No module named 'upfirdn2d_plugin'",
  "ModuleNotFoundError: No module named 'upfirdn2d_plugin'",
  "ModuleNotFoundError: No module named 'upfirdn2d_plugin'",
  "ModuleNotFoundError: No module named 'upfirdn2d_plugin'"]}

## Paso 9 · Proyección de costo (escenarios)

Los objetivos de duración son **escenarios para decidir**, no recomendaciones cerradas: la duración final se fijará mirando la evolución de las métricas en validación.
Se incluye también el costo de **generar** 3060 imágenes por técnica (mismo tamaño que train).

In [16]:
filas = []
def fila(tecnica, escenario, horas):
    filas.append({"tecnica": tecnica, "escenario": escenario, "horas_L4": round(horas, 1),
                  "unidades": round(horas * UNITS_PER_HOUR, 1)})

N_TRAIN = len(train_df)
v = RESULTADOS["vae"]
for ep in [200, 500]:
    fila("VAE", f"{ep} épocas", ep * N_TRAIN / v["img_por_s"] / 3600)

d = RESULTADOS["difusion"]
for pasos in [50_000, 100_000, 150_000]:
    fila("Difusión", f"{pasos//1000}k pasos (batch {d['batch']})", pasos / d["pasos_por_s"] / 3600)
fila("Difusión", f"generar 3060 img ({DDIM_STEPS} pasos DDIM)", 3060 * d["muestreo_seg_por_img"] / 3600)

g = RESULTADOS["gan"]
if g["sec_por_kimg"]:
    for kimg in [1000, 2000, 3000]:
        fila("GAN", f"{kimg} kimg", kimg * g["sec_por_kimg"] / 3600)
else:
    print("⚠️ StyleGAN2-ADA no registró ticks completos: revisa el log antes de proyectar su costo.")

COSTOS = pd.DataFrame(filas)
COSTOS

⚠️ StyleGAN2-ADA no registró ticks completos: revisa el log antes de proyectar su costo.


,tecnica,escenario,horas_L4,unidades
0,VAE,200 épocas,0.1,0.2
1,VAE,500 épocas,0.2,0.4
2,Difusión,50k pasos (batch 32),8.8,15.1
3,Difusión,100k pasos (batch 32),17.6,30.1
4,Difusión,150k pasos (batch 32),26.4,45.2
5,Difusión,generar 3060 img (50 pasos DDIM),0.6,1.1


## Paso 10 · Guardar el registro del piloto

In [17]:
log03 = {
    "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    "versiones": {"python": sys.version.split()[0], "torch": torch.__version__, "cuda_torch": torch.version.cuda,
                  "diffusers": diffusers.__version__, "stylegan2_ada_commit": SG_COMMIT},
    "resolucion": RES, "units_per_hour": UNITS_PER_HOUR,
    "datasets": {"train_zip": SG_ZIP.name, "train_sha256": SHA_TRAIN, "val_zip": VAL_ZIP.name, "val_sha256": SHA_VAL,
                 "n_train": len(train_df), "n_val": len(val_df)},
    "decisiones": {"condicional": True, "flip_horizontal": False, "metricas_stylegan": "none",
                   "difusion": "UNet2DModel (64,128,256,256,512) + DDPM coseno; CFG p_uncond=0.1"},
    "piloto": RESULTADOS,
    "proyeccion_costos": COSTOS.to_dict(orient="records"),
}
(DIRS["logs"] / "03_piloto.json").write_text(json.dumps(log03, indent=2, ensure_ascii=False, default=str))
print(json.dumps(log03["piloto"], indent=2, ensure_ascii=False, default=str))

{
  "vae": {
    "pasos": 8383,
    "segundos": 300.0,
    "pasos_por_s": 27.943,
    "img_por_s": 1788.3,
    "batch": 64,
    "vram_pico_gb": 1.1,
    "perdida_inicio": 13394.6385,
    "perdida_final": 1357.4357,
    "parametros_M": 14.5
  },
  "difusion": {
    "pasos": 947,
    "segundos": 600.0,
    "pasos_por_s": 1.578,
    "img_por_s": 50.5,
    "batch": 32,
    "vram_pico_gb": 11.48,
    "perdida_inicio": 0.2584,
    "perdida_final": 0.0559,
    "parametros_M": 72.7,
    "muestreo_ddim_pasos": 50,
    "muestreo_seg_por_img": 0.754
  },
  "gan": {
    "ticks_registrados": 0,
    "kimg_alcanzado": 0.0,
    "sec_por_kimg": null,
    "img_por_s": null,
    "vram_pico_gb": null,
    "kernels_cuda_fallback": [
      "bias_act",
      "upfirdn2d"
    ],
    "gradfix_desactivado": [
      "conv2d_gradfix",
      "grid_sample_gradfix"
    ],
    "errores": [
      "ModuleNotFoundError: No module named 'bias_act_plugin'",
      "ModuleNotFoundError: No module named 'upfirdn2d_plugin'",
 

### Qué enviar al asesor
1. La salida del Paso 10 (resumen del piloto) y la tabla del Paso 9.
2. Si StyleGAN2-ADA falló o cayó a la implementación de referencia: las últimas líneas del log (Paso 8).
3. Las mallas `eda/piloto/*.png` son solo verificación de que el flujo funciona (tras pocos minutos se verán borrosas o como ruido).